<div dir="rtl" align="right">

# بثُّ LSL - عرضُ المُرسِلِ والمُستقبِلِ

**مجموعةُ البياناتِ**: PhysioNet Auditory EEG (Abo Alzahab et al., 2021)  
**القنواتُ**: P4, Cz, F8, T7  
**معدّلُ أخذِ العيناتِ**: 200 Hz  
**المُشاركُ**: 1

---

## نظرةٌ عامّةٌ

يُوضّحُ هذا الدفترُ إنشاءَ بثِّ **Lab Streaming Layer (LSL)** واستقبالَه. نُنشئُ مُرسِلَ بثِّ EEG مُحاكىً، نُرسلُ بياناتِ 4 قنواتٍ في حزمٍ، ثمَّ نُنشئُ مُستقبِلًا لاستقبالِ البياناتِ مُجددًا. إذا لم تكن مكتبةُ pylsl متاحةً (شائعٌ في Colab)، نَلجأُ إلى محاكاةٍ تَنسخُ البياناتَ مباشرةً.

## ماذا يَفعلُ هذا الدفترُ

1. يحمّلُ جميعَ القنواتِ الأربعِ من مجموعةِ بياناتِ EEG المحليةِ
2. يُنشئُ مُرسِلَ LSL (الاسمُ = "SimulatedEEG"، النوعُ = "EEG"، 4 قنواتٍ، 200 Hz)
3. يُرسلُ البياناتِ في حزمٍ من 50 عينةٍ إلى المُرسِلِ
4. يُنشئُ مُستقبِلًا ويستقبلُ البياناتِ مُجددًا
5. يَلجأُ إلى المحاكاةِ إذا لم تكن pylsl متاحةً

## المُخرجاتُ المُتوقّعةُ

- المُخطّطُ العلويُّ يُظهرُ **الإشارةَ الأصليةَ ذاتَ 4 قنواتٍ** (أولُ 5000 عينةٍ، مع إزاحةٍ للوضوحِ)
- المُخطّطُ السفليُّ يُظهرُ **الإشارةَ المُستقبَلةَ عبرَ LSL** (أولُ 5000 عينةٍ، مع إزاحةٍ للوضوحِ)
- إذا استُخدمَ LSL الحقيقيُّ، يجبُ أن تكونَ الإشاراتُ مُتطابقةً
- إذا استُخدمَ وضعُ المحاكاةِ، تُنسخُ البياناتُ مباشرةً (مُتطابقةٌ أيضًا)

## المُعاملاتُ الأساسيةُ

| المُعاملُ | القيمةُ | الوصفُ |
| --------- | ------- | ------ |
| FS | 200 Hz | معدّلُ أخذِ العيناتِ |
| CHANNEL_COUNT | 4 | عددُ قنواتِ EEG |
| CHUNK_SIZE | 50 | العيناتُ في كلِّ حزمةِ إرسالٍ |
| N_PLOT | 5000 | عددُ العيناتِ للرسمِ |
| اسمُ البثِّ | SimulatedEEG | مُعرّفُ بثِّ LSL |

</div>

<div dir="rtl" align="right">

## 1. تثبيتُ المكتباتِ

</div>

In [ ]:
!pip install scipy numpy plotly wfdb pylsl

<div dir="rtl" align="right">

## 2. استنساخُ المستودعِ وتنزيلُ بياناتِ مُشاركٍ واحدٍ

</div>

In [ ]:
import os
if not os.path.exists('python-EEG-Arabic-Resources'):
    !git clone https://github.com/NibrasAz7/python-EEG-Arabic-Resources.git
os.chdir('python-EEG-Arabic-Resources')

In [ ]:
from pathlib import Path
data_dir = Path('data/local')
if not data_dir.exists() or not any(data_dir.glob('*.dat')):
    !python data/download_local.py --output data/local --subjects 1

<div dir="rtl" align="right">

## 3. تحميلُ إشارةِ EEG

نحمّلُ تسجيلَ المُشاركِ 1 في التجربةِ 1، الجلسةِ 2، جميعَ القنواتِ الأربعِ (P4, Cz, F8, T7).

</div>

In [ ]:
import numpy as np
from utils.eeg_loader import load_local_eeg

timestamps, eeg_data, ch_names = load_local_eeg(
    data_dir='data/local', subject=1, experiment=1, session=2
)
fs = 200

print(f'Channels: {ch_names}')
print(f'Signal shape: {eeg_data.shape}')
print(f'Duration: {eeg_data.shape[0]/fs:.1f} seconds')

<div dir="rtl" align="right">

## 4. تطبيقُ المعالجةِ

نُنشئُ مُرسِلَ LSL، نُرسلُ بياناتِ 4 قنواتٍ في حزمٍ من 50 عينةٍ، ثمَّ نُنشئُ مُستقبِلًا لاستقبالِ البياناتِ مُجددًا. إذا لم تكن pylsl متاحةً، نَلجأُ إلى محاكاةٍ.

</div>

In [ ]:
CHUNK_SIZE = 50
CHANNEL_COUNT = 4
N_PLOT = 5000

used_real_lsl = False
received_data = None

try:
    from pylsl import StreamInfo, StreamOutlet, StreamInlet, resolve_byprop
    import time

    info = StreamInfo(
        name='SimulatedEEG', type='EEG',
        channel_count=CHANNEL_COUNT, nominal_srate=fs,
        channel_format='float32',
    )
    outlet = StreamOutlet(info)

    n_samples = eeg_data.shape[0]
    for start in range(0, n_samples, CHUNK_SIZE):
        end = min(start + CHUNK_SIZE, n_samples)
        chunk = eeg_data[start:end].astype(np.float32)
        outlet.push_chunk(chunk.tolist())

    time.sleep(0.5)

    streams = resolve_byprop('name', 'SimulatedEEG', timeout=2)
    inlet = StreamInlet(streams[0])

    received = []
    total_received = 0
    while total_received < n_samples:
        chunk_data, _ = inlet.pull_chunk(timeout=1.0)
        if not chunk_data:
            break
        received.extend(chunk_data)
        total_received += len(chunk_data)

    received_data = np.array(received[:n_samples])
    used_real_lsl = True
    print('Real LSL stream used successfully.')
except Exception as e:
    print(f'LSL not available ({e}), falling back to simulation.')
    received_data = eeg_data.copy()
    used_real_lsl = False
    print('Simulation mode used (data copied directly).')

print(f'Received data shape: {received_data.shape}')

<div dir="rtl" align="right">

## 5. رسمٌ تفاعليٌّ

**علامَ تُلاحظُ؟**
- المُخطّطُ العلويُّ يُظهرُ الإشارةَ الأصليةَ ذاتَ 4 قنواتٍ، مُزاحةً عموديًّا للوضوحِ
- المُخطّطُ السفليُّ يُظهرُ الإشارةَ المُستقبَلةَ عبرَ LSL (أو المحاكاةِ)، مُزاحةً أيضًا
- يجبُ أن يَبدوَ المُخطّطانِ **مُتطابقينِ** — يَحفظُ LSL البياناتِ بدقّةٍ
- كلُّ قناةٍ مُزاحةٌ بإزاحةٍ ثابتةٍ حتى لا تتداخلَ

</div>

In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

n_plot = min(N_PLOT, eeg_data.shape[0])
x = np.arange(n_plot)
offsets = [0, 100, 200, 300]
mode_label = 'Real LSL' if used_real_lsl else 'Simulation'

fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    subplot_titles=('Original 4-Channel Signal',
                                    f'Received via LSL ({mode_label})'))

for i in range(CHANNEL_COUNT):
    fig.add_trace(go.Scatter(x=x, y=eeg_data[:n_plot, i] + offsets[i],
                             name=f'{ch_names[i]} (orig)',
                             line=dict(width=0.5)), row=1, col=1)
    fig.add_trace(go.Scatter(x=x, y=received_data[:n_plot, i] + offsets[i],
                             name=f'{ch_names[i]} (recv)',
                             line=dict(width=0.5)), row=2, col=1)

fig.update_layout(height=700, title_text='LSL Stream - Outlet and Inlet Demo',
                  xaxis2_title='Sample index',
                  yaxis_title='Amplitude + offset (uV)',
                  yaxis2_title='Amplitude + offset (uV)')
fig.show()

<div dir="rtl" align="right">

## خلاصةٌ

- **LSL (Lab Streaming Layer)** بروتوكولٌ لبثِّ بياناتِ EEG في الوقتِ الحقيقيِّ بين التطبيقاتِ
- **المُرسِلُ** يَنشرُ البياناتِ، و**المُستقبِلُ** يَستقبلُها — يمكنُ أن يكونَا في عمليّاتٍ أو أجهزةٍ مُختلفةٍ
- تُرسَلُ البياناتُ في **حزمٍ** للكفاءةِ، لا عينةً واحدةً في كلِّ مرّةٍ
- عند عدمِ توفّرِ pylsl، تُوضّحُ **محاكاةٌ بديلةٌ** نفسَ المفهومِ
- يُستخدمُ LSL على نطاقٍ واسعٍ في أبحاثِ BCI لربطِ عتادِ EEG ببرامجِ التحليلِ

</div>